In [2]:
# CELL 1: Verify the active Python environment path
import sys

print("==================================================")
print("🔍 Active Python Executable Path:")
print(sys.executable)
print("==================================================")


🔍 Active Python Executable Path:
/Users/nileshkumar/Documents/personal/Study/Mtech/sem1/ML/Assignment2/MLAssignment2/venv/bin/python


In [3]:
# CELL 1: Import core data libraries and load the dataset
import pandas as pd
import numpy as np

# Load the dataset using your root file path
file_path = 'IT_customer_churn.csv'
df = pd.read_csv(file_path)

# Verify assignment dataset criteria (>=12 features, >=500 rows)
rows, columns = df.shape
print("=========================================")
print("✅ Dataset Loaded Successfully!")
print(f"📊 Number of Rows (Instances): {rows}")
print(f"📋 Number of Columns (Features + Target): {columns}")
print("=========================================")

# Display column names, data types, and null counts
print("\nDataset Structural Information:")
df.info()


✅ Dataset Loaded Successfully!
📊 Number of Rows (Instances): 7043
📋 Number of Columns (Features + Target): 20

Dataset Structural Information:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Cont

In [4]:
# CELL 2: Fix data types and handle missing values
print("--- Cleaning Data Types ---")

# Convert TotalCharges to numeric, turning empty spaces (" ") into NaN values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many missing values were generated by this conversion
missing_total_charges = df['TotalCharges'].isnull().sum()
print(f"Missing values found in TotalCharges: {missing_total_charges}")

# Fill NaN values in TotalCharges with 0.0 (since tenure is 0 for these records)
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Double-check that all missing values are taken care of across the dataframe
print("\nFinal missing value counts per column:")
print(df.isnull().sum())


--- Cleaning Data Types ---
Missing values found in TotalCharges: 11

Final missing value counts per column:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [5]:
# CELL 3: Check target class distribution
print("--- Target Class Distribution ---")
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

for idx in churn_counts.index:
    print(f"Class '{idx}': {churn_counts[idx]} rows ({churn_percentages[idx]:.2f}%)")


--- Target Class Distribution ---
Class 'No': 5174 rows (73.46%)
Class 'Yes': 1869 rows (26.54%)


In [6]:
# CELL 4: Encode categorical features and target variable
print("--- Encoding Categorical Variables ---")

# 1. Binary encode the target column 'Churn' (Yes -> 1, No -> 0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Identify remaining categorical columns (excluding target)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical features to encode: {categorical_cols}")

# 3. Apply One-Hot Encoding to categorical features
# drop_first=True prevents multi-collinearity by dropping one redundant category flag
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Convert any newly created boolean columns (True/False) into 1/0 for our ML models
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"\nFinal encoded dataset shape: {df_encoded.shape}")
print("All columns successfully converted to numeric types!")


--- Encoding Categorical Variables ---
Categorical features to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Final encoded dataset shape: (7043, 31)
All columns successfully converted to numeric types!


/var/folders/82/ffp01ktn5kn1bywj_w277pyh0000gn/T/ipykernel_13152/991003905.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [7]:
# CELL 5: Train-Test Split and Export Clean Test Data for Streamlit
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Perform an 80/20 stratified split to preserve our 73/27 class imbalance ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Combine X_test and y_test back together to create a true, realistic test upload file
test_df = X_test.copy()
test_df['Churn'] = y_test

# Save the test slice directly to your root directory as required by the assignment
test_df.to_csv('test_data.csv', index=False)

print("=========================================")
print("✅ Data Split Complete & Test File Saved!")
print(f"🏋️ Training Set Features Shape: {X_train.shape}")
print(f"🧪 Test Set Features Shape: {X_test.shape}")
print("📁 Generated: 'test_data.csv' in your project root.")
print("=========================================")


✅ Data Split Complete & Test File Saved!
🏋️ Training Set Features Shape: (5634, 30)
🧪 Test Set Features Shape: (1409, 30)
📁 Generated: 'test_data.csv' in your project root.


In [8]:
# CELL 6: Scale numerical features for distance-based models
from sklearn.preprocessing import StandardScaler

# Identify the columns that need scaling (our original continuous variables)
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Initialize the StandardScaler
scaler = StandardScaler()

# Create copies of our splits to avoid modifying the original dataframes
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit the scaler on the training set and transform both sets
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Save the fitted scaler so we can use it inside our Streamlit application later
import joblib
import os

os.makedirs('model', exist_ok=True)
joblib.dump(scaler, 'model/scaler.joblib')

print("=========================================")
print("✅ Continuous Features Scaled Successfully!")
print("📁 Saved: 'model/scaler.joblib' for app deployment.")
print("=========================================")


✅ Continuous Features Scaled Successfully!
📁 Saved: 'model/scaler.joblib' for app deployment.


In [9]:
# CELL 7: Define a reusable function to compute all 6 assignment metrics and save models
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib

# Dictionary to hold the final performance metrics of all models
evaluation_results = {}

def evaluate_and_save_model(model_name, model_object, X_tr, X_te, y_tr, y_te):
    """
    Fits the model, predicts on the test set, calculates all 6 required 
    assignment metrics, and saves the trained model artifact to the model/ folder.
    """
    # 1. Train the model
    model_object.fit(X_tr, y_tr)
    
    # 2. Generate predictions
    y_pred = model_object.predict(X_te)
    
    # Generate probability scores for the AUC calculations
    if hasattr(model_object, "predict_proba"):
        y_prob = model_object.predict_proba(X_te)[:, 1]
    else:
        # Fallback for models without predict_proba (like some configurations of SVM)
        y_prob = model_object.decision_function(X_te)
        
    # 3. Calculate all 6 required evaluation metrics
    metrics = {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "AUC":       roc_auc_score(y_te, y_prob),
        "Precision": precision_score(y_te, y_pred),
        "Recall":    recall_score(y_te, y_pred),
        "F1":        f1_score(y_te, y_pred),
        "MCC":       matthews_corrcoef(y_te, y_pred)
    }
    
    # Store metrics in our global results tracker
    evaluation_results[model_name] = metrics
    
    # 4. Save the trained model artifact as a .joblib file
    model_path = f"model/{model_name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model_object, model_path)
    
    # 5. Print a clean, formatted readout of the performance
    print(f"================ {model_name} ================")
    for metric_name, score in metrics.items():
        print(f"🎯 {metric_name:<10}: {score:.4f}")
    print(f"📁 Saved Model to: {model_path}")
    print("==================================================\n")

print("✅ Metric evaluation and model-saving routine defined successfully!")


✅ Metric evaluation and model-saving routine defined successfully!


In [10]:
# CELL 8: Train and Evaluate Logistic Regression
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression with a standard max_iter to guarantee convergence
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Pass our scaled training and test datasets into our custom evaluation routine
evaluate_and_save_model(
    model_name="Logistic Regression",
    model_object=log_reg,
    X_tr=X_train_scaled,
    X_te=X_test_scaled,
    y_tr=y_train,
    y_te=y_test
)


================ Logistic Regression ================
🎯 Accuracy  : 0.8062
🎯 AUC       : 0.8422
🎯 Precision : 0.6593
🎯 Recall    : 0.5588
🎯 F1        : 0.6049
🎯 MCC       : 0.4806
📁 Saved Model to: model/logistic_regression.joblib



In [11]:
# CELL 9: Train and Evaluate Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

# Initialize Decision Tree with max_depth to control overfitting
dt_classifier = DecisionTreeClassifier(max_depth=6, random_state=42)

# Evaluate and save the model
evaluate_and_save_model(
    model_name="Decision Tree",
    model_object=dt_classifier,
    X_tr=X_train_scaled,
    X_te=X_test_scaled,
    y_tr=y_train,
    y_te=y_test
)


================ Decision Tree ================
🎯 Accuracy  : 0.8006
🎯 AUC       : 0.8219
🎯 Precision : 0.6667
🎯 Recall    : 0.4973
🎯 F1        : 0.5697
🎯 MCC       : 0.4515
📁 Saved Model to: model/decision_tree.joblib



In [12]:
# CELL 10: Train and Evaluate K-Nearest Neighbors Classifier
from sklearn.neighbors import KNeighborsClassifier

# Initialize KNN Classifier with 7 neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=7)

# Evaluate and save the model
evaluate_and_save_model(
    model_name="KNN",
    model_object=knn_classifier,
    X_tr=X_train_scaled,
    X_te=X_test_scaled,
    y_tr=y_train,
    y_te=y_test
)


================ KNN ================
🎯 Accuracy  : 0.7708
🎯 AUC       : 0.8033
🎯 Precision : 0.5706
🎯 Recall    : 0.5508
🎯 F1        : 0.5605
🎯 MCC       : 0.4057
📁 Saved Model to: model/knn.joblib



In [13]:
# CELL 11: Train and Evaluate Naive Bayes Classifier
from sklearn.naive_bayes import GaussianNB

# Initialize Gaussian Naive Bayes Classifier
nb_classifier = GaussianNB()

# Evaluate and save the model
evaluate_and_save_model(
    model_name="Naive Bayes",
    model_object=nb_classifier,
    X_tr=X_train_scaled,
    X_te=X_test_scaled,
    y_tr=y_train,
    y_te=y_test
)


================ Naive Bayes ================
🎯 Accuracy  : 0.6558
🎯 AUC       : 0.8092
🎯 Precision : 0.4269
🎯 Recall    : 0.8663
🎯 F1        : 0.5719
🎯 MCC       : 0.3951
📁 Saved Model to: model/naive_bayes.joblib



In [14]:
# CELL 12: Train and Evaluate Support Vector Machine
from sklearn.svm import SVC

# Initialize SVM with RBF kernel and enable probability estimates for AUC calculation
svm_classifier = SVC(kernel='rbf', probability=True, random_state=42)

# Evaluate and save the model
evaluate_and_save_model(
    model_name="Support Vector Machine",
    model_object=svm_classifier,
    X_tr=X_train_scaled,
    X_te=X_test_scaled,
    y_tr=y_train,
    y_te=y_test
)


/Users/nileshkumar/Documents/personal/Study/Mtech/sem1/ML/Assignment2/MLAssignment2/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


================ Support Vector Machine ================
🎯 Accuracy  : 0.7949
🎯 AUC       : 0.7948
🎯 Precision : 0.6545
🎯 Recall    : 0.4813
🎯 F1        : 0.5547
🎯 MCC       : 0.4339
📁 Saved Model to: model/support_vector_machine.joblib



In [16]:
# CELL 13: Compile evaluation metrics into the final assignment comparison table
# Create a structured DataFrame from our evaluation dictionary
comparison_df = pd.DataFrame(evaluation_results).T

# Reorder columns to precisely match the assignment template layout
assignment_metrics_order = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison_df = comparison_df[assignment_metrics_order].round(4)

print("=========================================================================")
print("📊 FINAL ASSIGNMENT MODEL COMPARISON TABLE")
print("=========================================================================")
print(comparison_df)
print("=========================================================================\n")

print("📋 Copy-paste Text Layout for your GitHub README.md:")
print("| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |")
print("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |")
for model, row in comparison_df.iterrows():
    print(f"| {model} | {row['Accuracy']} | {row['AUC']} | {row['Precision']} | {row['Recall']} | {row['F1']} | {row['MCC']} |")


📊 FINAL ASSIGNMENT MODEL COMPARISON TABLE
                        Accuracy     AUC  Precision  Recall      F1     MCC
Logistic Regression       0.8062  0.8422     0.6593  0.5588  0.6049  0.4806
Decision Tree             0.8006  0.8219     0.6667  0.4973  0.5697  0.4515
KNN                       0.7708  0.8033     0.5706  0.5508  0.5605  0.4057
Naive Bayes               0.6558  0.8092     0.4269  0.8663  0.5719  0.3951
Support Vector Machine    0.7949  0.7948     0.6545  0.4813  0.5547  0.4339

📋 Copy-paste Text Layout for your GitHub README.md:
| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Logistic Regression | 0.8062 | 0.8422 | 0.6593 | 0.5588 | 0.6049 | 0.4806 |
| Decision Tree | 0.8006 | 0.8219 | 0.6667 | 0.4973 | 0.5697 | 0.4515 |
| KNN | 0.7708 | 0.8033 | 0.5706 | 0.5508 | 0.5605 | 0.4057 |
| Naive Bayes | 0.6558 | 0.8092 | 0.4269 | 0.8663 | 0.5719 | 0.3951 |
| Support Vector Machine | 0.7949 | 0.7948 | 0.65